# PhysAR-IDS -- NB1 (v6base25, major-revision run): Data, Baselines, Encoders -- multiseed, 25-FEATURE VARIANT

**Dataset:** Sweeten et al. (2025) SmartGrid-CyberPhysical-Attack-Dataset (doi:10.21227/symr-bz19)

> **THIS IS THE 25-FEATURE VARIANT (`v6base25`)** -- it uses the 11 raw physical telemetry features
> (`Freq, Theta, V_A, V_B, V_C, I_A, I_B, I_C, ActivePower, ReactivePower, BreakerStatus`) and the 10 original cyber features
> the 4 physics-derived features (`voltage_imbalance, current_imbalance, power_factor, freq_deviation`) and the 10 original cyber features
> (1 s window). No temporal or extended cyber features enter the model matrix (`Freq`, `BreakerStatus` and `freq_deviation`
> are constant, so 22 features carry information). It runs side by side with the 93-feature `v6` notebook: same data, same
> split, same seeds; artefacts are tagged `v6base25` so nothing is overwritten. Section 6e is reduced to this one feature set,
> and the `v5_feature_set` / `legacy10` ablation groups are dropped. In design experiments (burst-level emulation, 5 seeds)
> this is the original base feature set (25) that sits between the 21-feature and 93-feature runs.
>
> The text below describes the shared pipeline (the extended cyber columns are still *computed* by the join, but not used).

**Pipeline version `v6` = v4 + two changes** (temporal split, `TEST_FRAC=0.15`, 6 classes unchanged):

1. **Power-factor bug fix (from v5).**
The fix: `ActivePower` is parsed as `int64`, so `ActivePower**2` **overflowed** and the derived `power_factor` was
physically impossible (|PF| > 1) in **58.9% of rows**. It is now computed in float64 (`PF_FIX = True`). Set `PF_FIX = False`
(v6 requires the fix.)
2. **Extended cyber features (new in v6).** The physical stream arrives in bursts ~5 s apart (~229 rows share one timestamp
   and one identical feature vector), but the original cyber window covered only the last **1 s** of each 5 s gap, i.e. ~20% of
   the traffic. v6 keeps the original 10 cyber features unchanged and **adds 50 more**: 25 summary statistics (packet/byte
   counts, length min/max/sum, distinct IPs/MACs/ports, port entropy, protocol shares, inter-arrival stats, PLC-outbound share)
   computed over a **2 s** and a **5 s** causal window. The windows are computed once per distinct (PLC, timestamp) instead of
   once per row (~200x faster). Window length and feature set were chosen by time-blocked cross-validation **inside the
   training data only** (Section 6e); the test set was not used for selection.

All artefacts are tagged `v6`, so earlier versions never get overwritten, but **every downstream notebook must be pointed at
`v6`**; the original 43 features keep their positions (0-42) and the 50 new cyber columns are appended (43-92), so use
`CYBER_IDX` stored in `phase1_data_v6.pkl` for the attacker-controllable set (all 60 `cyber_*` columns). What else is new in this run:

| Reviewer point (Major Revision) | Where it is handled in this notebook |
|---|---|
| Main results had no error bars / repeated runs | Sections 8-10: **every model is run on 5 seeds** (42, 7, 123, 2024, 99); tables report mean +/- std and a 95% t-interval |
| Wilcoxon with n=5 cannot reach p<0.05 (min two-sided p = 0.0625) | Section 11: paired mean difference +/- std and paired t-test; Wilcoxon deliberately not used as a significance test |
| Report class sizes | Section 6b: per-class train/test counts saved to `nb1_class_sizes_v4.csv` |
| Possible leakage / session artefacts; "why would ransomware/reverse-shell touch voltage or current?" | Section 6c: per-class recording-session audit and per-class physical-state summary |
| "Physics immunity" may be a consequence of the threat model, not of physics | Section 9: feature-group ablation (physical-only / cyber-only / both), per class, 5 seeds -- shows which feature group each class is actually detected from |
| GAT F1 of 0.048 is below chance and suggests a bug | Section 13: chance-level reference, flat-feature control on the same graphs, and a masked GAT variant that fixes the empty-node scaling/attention issue |
| RF / Transformer beat XGBoost -- say so openly | Section 12: explicit ranking with mean +/- std and a printed statement of which models beat XGBoost |
| 5% false-alarm rate on grid traffic | Section 8: benign false-positive rate and alerts per PLC-hour for every model |

Seeds change model initialisation / subsampling / shuffling only. The split is deterministic by construction.

**Runtime:** local GPU (Phase 1 XGBoost/RF/LR run on CPU for comparability with earlier v4 numbers; AE and encoders use CUDA).

## 0. Setup

In [ ]:
import os, sys, random, time, json, platform, warnings, zipfile
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------------------
# PATHS (Windows, local). Dataset zip sits one level above the output folder.
# ----------------------------------------------------------------------------------
BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')   # ALL pkl / csv / json results are saved here
ZIP_PATH    = os.path.join(BASE_DIR, 'SmartGrid-CyberPhysical-Attack-Dataset.zip')
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')

os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

PF_FIX           = True          # True: float64 power-factor (v5). False: legacy int64-overflow behaviour (v4)
PIPELINE_VERSION = 'v6base25'    # 25-feature variant: 11 raw physical + 4 derived + 10 cyber (1 s window); runs in parallel with v6 (93 features)
RUN_TAG          = f'{PIPELINE_VERSION}_multiseed'
SEEDS            = [42, 7, 123, 2024, 99]   # same seed set used in the earlier robustness checks
RANDOM_STATE     = SEEDS[0]


def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Working directory : {os.getcwd()}')
print(f'Zip path          : {ZIP_PATH}  (exists={os.path.exists(ZIP_PATH)})')
print(f'Device            : {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == "cuda" else ''))
print(f'Seeds             : {SEEDS}')
print(f'PF_FIX={PF_FIX} -> PIPELINE_VERSION={PIPELINE_VERSION}')
assert PF_FIX, 'v6base25 requires PF_FIX=True (the legacy int64 overflow is not supported in v6)'
if DEVICE.type != 'cuda':
    print('WARNING: CUDA not available -- AE/encoder sections will be slow. Check your PyTorch build.')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import xgboost as xgb
from scipy import stats
import joblib
print('xgboost', xgb.__version__, '| torch', torch.__version__)

## 1. Extract dataset (skipped if already extracted)

In [ ]:
CLASSES = {
    'benign': 0, 'backdoor': 1, 'Bruteforce': 2, 'FDI': 3, 'ransomware': 4, 'reverseshell': 5,
}
CYBER_FILE = {
    'benign': 'Cyber.csv', 'backdoor': 'Cyber.csv', 'Bruteforce': 'Cyber.csv',
    'FDI': 'Cyber.csv', 'ransomware': 'Cyber.csv',
    'reverseshell': 'cyber.csv',  # NOTE: lowercase for this class only
}

def _extracted(cls_name):
    return all(os.path.exists(os.path.join(DATA_DIR, cls_name, f))
               for f in [CYBER_FILE[cls_name], 'Physical.csv'])

if all(_extracted(c) for c in CLASSES):
    print('Dataset already extracted -- skipping.')
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        for cls_name in CLASSES:
            os.makedirs(os.path.join(DATA_DIR, cls_name), exist_ok=True)
            for fname in [CYBER_FILE[cls_name], 'Physical.csv']:
                src = f'Dataset/{cls_name}/{fname}'
                with z.open(src) as fsrc, open(os.path.join(DATA_DIR, cls_name, fname), 'wb') as fdst:
                    fdst.write(fsrc.read())
    print('Extraction complete.')

## 2. Loading + cleaning helpers

The `Physical.csv` numeric columns (V_A/B/C, I_A/B/C, ActivePower, ReactivePower) are stored as
comma-separated strings (e.g. `"85,732,060"`) and must be cleaned before use.

**`FDI`'s `Cyber.csv` is missing its real `node` (PLCID) data column entirely.** Every field
is shifted one position left relative to the header: what the header calls `node` is
actually an epoch timestamp (median ~1.68e9), `frame.time_epoch` actually holds a MAC
address, and so on down the row. This was confirmed directly against the raw file --
`node`'s median value is in the billions, nowhere near a PLCID 1-9.

Simply re-aligning the columns and setting `node = NaN` (the naive fix) would silently
zero-fill FDI's entire cyber-feature set, since nothing would ever match a PLCID. Instead,
we recover the true PLCID from the IP subnet, which is fixed network infrastructure and
holds identically across every class we checked (benign, ransomware, reverseshell):

| PLCID (`node`) | Subnet |
|---|---|
| 1 | 192.168.3.x | 2 | 192.168.4.x | 3 | 192.168.5.x |
| 4 | 192.168.6.x | 5 | 192.168.7.x | 6 | 192.168.8.x |
| 7 | 192.168.9.x | 8 | 192.168.10.x | 9 | 192.168.11.x |

i.e. `PLCID = third_octet - 2`, for third octets 3-11. `192.168.200.x` (SCADA/HMI) and
`192.168.23.x` (shared historian/gateway) aren't PLC-specific and correctly resolve to
`NaN` (no PLCID) under this rule.

`ransomware`'s `Cyber.csv` also has a harmless trailing all-null `Unnamed: 12` column
(confirmed directly against the raw file) -- dropped, not treated as data.

In [ ]:
NUMERIC_PHYS_COLS = ['Freq', 'Theta', 'V_A', 'V_B', 'V_C',
                      'I_A', 'I_B', 'I_C', 'ActivePower', 'ReactivePower']


def clean_numeric(series):
    """Strip thousands-separator commas and cast to float."""
    if series.dtype == object or pd.api.types.is_string_dtype(series):
        return pd.to_numeric(series.astype(str).str.replace(',', '', regex=False),
                              errors='coerce')
    return pd.to_numeric(series, errors='coerce')


def infer_plcid_from_ip(ip_series):
    """PLCID = subnet third octet - 2 (confirmed against benign/ransomware/reverseshell).
    192.168.200.x (SCADA/HMI) and 192.168.23.x (shared historian) correctly fall outside
    [1, 9] and resolve to NaN -- they are not PLC-specific."""
    third_octet = ip_series.astype(str).str.extract(r'^192\.168\.(\d+)\.')[0].astype(float)
    plcid = third_octet - 2
    return plcid.where(plcid.between(1, 9))


def load_cyber_raw(path):
    """Load a Cyber.csv, detecting and repairing the FDI-style column shift, and
    recovering PLCID from IP subnet when the real `node` column is unrecoverable."""
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    # Drop known harmless trailing empty column (e.g. ransomware's Cyber.csv)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

    node_numeric = pd.to_numeric(df['node'], errors='coerce')
    if node_numeric.median() > 1e6:
        # 'node' is really an epoch timestamp -> every field is shifted one column left.
        # Re-parse with the corrected 11-field layout, then recover PLCID from IP subnet
        # instead of leaving it unrecoverable.
        raw = pd.read_csv(path, header=None, skiprows=1, low_memory=False)
        base_cols = [
            'frame.time_epoch', 'eth.src', 'eth.dst', 'ip.src', 'ip.dst',
            '_ws.col.Protocol', 'ip.len', 'tcp.srcport', 'tcp.dstport',
            'udp.srcport', 'udp.dstport',
        ]
        raw = raw.iloc[:, :len(base_cols)]
        raw.columns = base_cols
        raw['node'] = infer_plcid_from_ip(raw['ip.src']).fillna(infer_plcid_from_ip(raw['ip.dst']))
        df = raw

    return df

## 3. Cyber-physical join (v6: per-burst windows, extended cyber features)

Physical samples are the anchor (one row per physical sample). For each PLCID, cyber packets from the matching `node` are
aggregated into **causal** windows ending at the sample's timestamp (packets before or at the timestamp only). Rows that share
a timestamp share an identical window, so windows are computed **once per distinct timestamp** and copied to the rows.
The first 10 `cyber_*` columns are the original 1 s features (verified identical to the v4/v5 values); the new `cyber_x2_*` /
`cyber_x5_*` columns are the 2 s and 5 s windows. Everything is causal: no packet after the sample's timestamp is used.

In [ ]:
EXT_WINDOWS = (2.0, 5.0)     # seconds; 5 s = spacing between physical bursts, so it covers ALL traffic since the last burst
PROTOS_EXT = ['TCP', 'Modbus/TCP', 'HTTP', 'TLSv1.2', 'ARP', 'SSHv2']
LEGACY_CYBER = ['cyber_pkt_count', 'cyber_pktlen_mean', 'cyber_pktlen_std', 'cyber_n_unique_srcport',
                'cyber_n_unique_dstport', 'cyber_modbus_frac', 'cyber_tls_frac', 'cyber_tcp_frac',
                'cyber_iat_mean', 'cyber_iat_std']
EXT_KEYS = (['n', 'len_sum', 'len_mean', 'len_std', 'len_max', 'len_min', 'n_src_ip', 'n_dst_ip', 'n_mac',
             'n_sport', 'n_dport', 'dport_ent', 'top_dport_share', 'iat_mean', 'iat_std', 'iat_max',
             'out_frac', 'small_frac'] + [f'p_{p}' for p in PROTOS_EXT] + ['p_other'])
EXT_COLS = [f'cyber_x{int(w)}_{k}' for w in EXT_WINDOWS for k in EXT_KEYS]


def _entropy(x):
    if len(x) == 0:
        return 0.0
    _, cnt = np.unique(x, return_counts=True)
    p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())


def _legacy_window(c, t, window=1.0):
    """Exactly the original 10 cyber features (1 s causal window)."""
    hi = np.searchsorted(c['t'], t, side='right'); lo = np.searchsorted(c['t'], t - window, side='left')
    if hi - lo == 0:
        return [0, 0.0, 0.0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0]
    wl, wp, ws, wd, wt = c['len'][lo:hi], c['proto_l'][lo:hi], c['sport'][lo:hi], c['dport'][lo:hi], c['t'][lo:hi]
    iat = np.diff(wt) if len(wt) > 1 else np.array([0.0])
    return [len(wt), np.nanmean(wl) if len(wl) else 0.0, np.nanstd(wl) if len(wl) else 0.0,
            len(np.unique(ws[~np.isnan(ws)])), len(np.unique(wd[~np.isnan(wd)])),
            float(np.mean(np.char.find(wp, 'modbus') >= 0)), float(np.mean(np.char.find(wp, 'tls') >= 0)),
            float(np.mean(np.char.find(wp, 'tcp') >= 0)),
            float(np.mean(iat)) if len(iat) else 0.0, float(np.std(iat)) if len(iat) else 0.0]


def _ext_window(c, t, window, plcid):
    hi = np.searchsorted(c['t'], t, side='right'); lo = np.searchsorted(c['t'], t - window, side='left')
    n = hi - lo
    if n == 0:
        return [0.0] * len(EXT_KEYS)
    L, tt, pr, dp, sp = c['len'][lo:hi], c['t'][lo:hi], c['proto'][lo:hi], c['dport'][lo:hi], c['sport'][lo:hi]
    dpv = dp[~np.isnan(dp)]
    iat = np.diff(tt) if n > 1 else np.array([0.0])
    shares = [float(np.mean(pr == p)) for p in PROTOS_EXT]
    return ([float(n), float(np.nansum(L)), float(np.nanmean(L)), float(np.nanstd(L)), float(np.nanmax(L)), float(np.nanmin(L)),
             float(len(np.unique(c['sip'][lo:hi]))), float(len(np.unique(c['dip'][lo:hi]))), float(len(np.unique(c['mac'][lo:hi]))),
             float(len(np.unique(sp[~np.isnan(sp)]))), float(len(np.unique(dpv))), _entropy(dpv),
             float(np.unique(dpv, return_counts=True)[1].max() / len(dpv)) if len(dpv) else 0.0,
             float(iat.mean()), float(iat.std()), float(iat.max()),
             float(np.mean(c['sip3'][lo:hi] == plcid + 2)), float(np.nanmean(L < 80))]
            + shares + [1.0 - sum(shares)])


def load_class(cls_name):
    cls_dir = os.path.join(DATA_DIR, cls_name)
    cyber = load_cyber_raw(os.path.join(cls_dir, CYBER_FILE[cls_name]))
    phys = pd.read_csv(os.path.join(cls_dir, 'Physical.csv'), low_memory=False)

    for col in NUMERIC_PHYS_COLS:
        phys[col] = clean_numeric(phys[col])
    phys['BreakerStatus'] = pd.to_numeric(phys['BreakerStatus'], errors='coerce')
    phys['@timestamp'] = pd.to_numeric(phys['@timestamp'], errors='coerce')
    phys['PLCID'] = pd.to_numeric(phys['PLCID'], errors='coerce')

    cyber['frame.time_epoch'] = pd.to_numeric(cyber['frame.time_epoch'], errors='coerce')
    cyber['node'] = pd.to_numeric(cyber['node'], errors='coerce')
    cyber['ip.len'] = clean_numeric(cyber['ip.len'])
    cyber['_ws.col.Protocol'] = cyber['_ws.col.Protocol'].astype(str)
    for k in ['tcp.srcport', 'tcp.dstport', 'udp.srcport', 'udp.dstport']:
        cyber[k] = clean_numeric(cyber[k])

    cyber = cyber.dropna(subset=['frame.time_epoch', 'node']).sort_values('frame.time_epoch', kind='stable')
    phys = phys.dropna(subset=['@timestamp', 'PLCID']).sort_values('@timestamp', kind='stable')

    phys_out = []
    for plcid, p_grp in phys.groupby('PLCID'):
        c_grp = cyber[cyber['node'] == plcid].sort_values('frame.time_epoch', kind='stable')
        p_grp = p_grp.sort_values('@timestamp', kind='stable').reset_index(drop=True)
        uts = np.unique(p_grp['@timestamp'].values)          # physical rows arrive in bursts sharing one timestamp
        if len(c_grp) == 0:
            tab = pd.DataFrame(0.0, index=uts, columns=LEGACY_CYBER + EXT_COLS)
        else:
            c = dict(
                t=c_grp['frame.time_epoch'].values, len=c_grp['ip.len'].values,
                proto=np.asarray(c_grp['_ws.col.Protocol'].tolist(), dtype='U40'),
                sport=c_grp['tcp.srcport'].fillna(c_grp['udp.srcport']).values,
                dport=c_grp['tcp.dstport'].fillna(c_grp['udp.dstport']).values,
                sip=pd.factorize(c_grp['ip.src'].astype(str))[0], dip=pd.factorize(c_grp['ip.dst'].astype(str))[0],
                mac=pd.factorize(c_grp['eth.src'].astype(str))[0],
                sip3=c_grp['ip.src'].astype(str).str.extract(r'^192\.168\.(\d+)\.')[0].astype(float).values)
            c['proto_l'] = np.char.lower(c['proto'])
            recs = []
            for t in uts:                                    # ~50 distinct timestamps per PLC, not ~12,000 rows
                row = _legacy_window(c, t)
                for w in EXT_WINDOWS:
                    row = row + _ext_window(c, t, w, plcid)
                recs.append(row)
            tab = pd.DataFrame(recs, index=uts, columns=LEGACY_CYBER + EXT_COLS)
        feat = tab.loc[p_grp['@timestamp'].values].reset_index(drop=True)
        phys_out.append(pd.concat([p_grp, feat], axis=1))

    out = pd.concat(phys_out, axis=0).reset_index(drop=True)
    out['label'] = CLASSES[cls_name]
    out['label_name'] = cls_name
    return out

## 4. Physics features (KVL/KCL)

All features computed **per-row** from that row's own raw telemetry -- no aggregation across
rows, so no leakage across the train/test split.

In [ ]:
def compute_physics_features(df):
    if PF_FIX:   # ActivePower is parsed as int64; squaring it overflows int64 -> impossible |power_factor| > 1
        for _c in ['V_A','V_B','V_C','I_A','I_B','I_C','ActivePower','ReactivePower']:
            df[_c] = df[_c].astype('float64')
    V_A, V_B, V_C = df['V_A'], df['V_B'], df['V_C']
    I_A, I_B, I_C = df['I_A'], df['I_B'], df['I_C']

    V_mean = (V_A + V_B + V_C) / 3.0
    I_mean = (I_A + I_B + I_C) / 3.0

    df['voltage_imbalance'] = (np.maximum.reduce([np.abs(V_A - V_mean),
                                                     np.abs(V_B - V_mean),
                                                     np.abs(V_C - V_mean)])
                                / V_mean.replace(0, np.nan))
    df['current_imbalance'] = (np.maximum.reduce([np.abs(I_A - I_mean),
                                                     np.abs(I_B - I_mean),
                                                     np.abs(I_C - I_mean)])
                                / I_mean.replace(0, np.nan))

    S = np.sqrt(df['ActivePower']**2 + df['ReactivePower']**2)
    df['power_factor'] = df['ActivePower'] / S.replace(0, np.nan)
    df['freq_deviation'] = np.abs(df['Freq'] - 60.0)

    df[['voltage_imbalance', 'current_imbalance', 'power_factor']] = \
        df[['voltage_imbalance', 'current_imbalance', 'power_factor']].fillna(0)

    return df

## 5. Run the full pipeline (all 6 classes)

In [ ]:
all_dfs = []
for cls_name in CLASSES:
    print(f'Loading {cls_name}...')
    df = load_class(cls_name)
    print(f'  -> {len(df)} rows')
    all_dfs.append(df)

full = pd.concat(all_dfs, axis=0).reset_index(drop=True)
print(f'\nTotal rows: {len(full)}')
print(full['label_name'].value_counts())

full = compute_physics_features(full)
print(f'\nJoined dataset shape (before optional temporal features): {full.shape}')

In [ ]:
pf_bad = float((full['power_factor'].abs() > 1 + 1e-9).mean())
print(f'Rows with |power_factor| > 1 (physically impossible): {pf_bad:.2%}   (ActivePower dtype: {full["ActivePower"].dtype})')
if PF_FIX:
    assert pf_bad == 0.0, 'power_factor still out of range -- overflow not fixed'
    print('OK: power_factor is within [-1, 1] for every row.')
else:
    print('PF_FIX=False: legacy v4 behaviour (overflowed power_factor) -- for reproducing old artefacts only.')

## 5b. Optional: temporal derivative + rolling-window features

Everything so far is **instantaneous, per-row** -- it can represent "what the signal is
right now" but not "the signal just jumped" or "the signal just got noisy," which is
exactly how several attack types manifest (transients rather than steady-state level
shifts). Two genuinely new feature types, computed **causally** (backward-looking only,
via `.diff()` / `.rolling()`), per `(label, PLCID)` group sorted by timestamp:

- **`d_X`, `d_X_dt`** -- first difference and rate of change, for `Freq`, `ActivePower`,
  `ReactivePower`, `Theta`, `V_avg`, `I_avg`.
- **`roll_mean_X`, `roll_std_X`** -- 5-sample rolling mean/std, for `ActivePower`,
  `Freq`, `V_avg`.

**Important grouping fix vs. the reference implementation this is adapted from:** the
diff/rolling must be grouped by `(label, PLCID)`, not `PLCID` alone. Each class (`label`)
is a *separate, non-contiguous recording session* -- grouping by `PLCID` only would
silently compute a "rate of change" between the last row of one class's PLCID-6 series
and the first row of a *different* class's PLCID-6 series after concatenation, which is
meaningless. Grouping by `(label, PLCID)` (matching the same grouping already used for
the temporal split and Phase 2 windows/graphs) avoids that.

**Ordering vs. the split:** this is computed on the *full, pre-embargo-drop* per-group
time series, then the temporal + embargo split (Section 6) runs afterward. That's
deliberate -- a rolling/diff feature for the first test-block row after the embargo gap
correctly reflects genuine recent physical history (which legitimately includes the
now-embargoed rows), rather than an artificial "cold start" at each block boundary.

**Toggle `USE_TEMPORAL_FEATURES` below to run with/without this block** for a direct
ablation against the existing feature set -- this is not a guaranteed F1 improvement,
just new information the model didn't have access to before; whether it helps is an
empirical question this toggle lets you answer directly.

In [ ]:
USE_TEMPORAL_FEATURES = False  # flip to False and re-run for a direct with/without ablation

if USE_TEMPORAL_FEATURES:
    full = full.sort_values(['label', 'PLCID', '@timestamp']).reset_index(drop=True)

    full['V_avg_tmp'] = full[['V_A', 'V_B', 'V_C']].mean(axis=1)
    full['I_avg_tmp'] = full[['I_A', 'I_B', 'I_C']].mean(axis=1)

    dt = full.groupby(['label', 'PLCID'])['@timestamp'].diff()
    dt_safe = dt.replace(0, np.nan)  # many consecutive rows share a timestamp (see Section 2)

    diff_pairs = [
        ('Freq', 'Freq'), ('ActivePower', 'ActivePower'), ('ReactivePower', 'ReactivePower'),
        ('Theta', 'Theta'), ('V_avg_tmp', 'V_avg'), ('I_avg_tmp', 'I_avg'),
    ]
    for src_col, out_name in diff_pairs:
        d = full.groupby(['label', 'PLCID'])[src_col].diff()
        full[f'd_{out_name}'] = d.fillna(0.0)
        full[f'd_{out_name}_dt'] = (d / dt_safe).fillna(0.0)

    roll_pairs = [('ActivePower', 'ActivePower'), ('Freq', 'Freq'), ('V_avg_tmp', 'V_avg')]
    for src_col, out_name in roll_pairs:
        full[f'roll_mean_{out_name}'] = full.groupby(['label', 'PLCID'])[src_col].transform(
            lambda s: s.rolling(5, min_periods=1).mean())
        full[f'roll_std_{out_name}'] = full.groupby(['label', 'PLCID'])[src_col].transform(
            lambda s: s.rolling(5, min_periods=1).std().fillna(0.0))

    full = full.drop(columns=['V_avg_tmp', 'I_avg_tmp'])

    TEMPORAL_FEATURE_COLS = (
        [f'd_{n}' for n in ['Freq', 'ActivePower', 'ReactivePower', 'Theta', 'V_avg', 'I_avg']]
        + [f'd_{n}_dt' for n in ['Freq', 'ActivePower', 'ReactivePower', 'Theta', 'V_avg', 'I_avg']]
        + [f'roll_mean_{n}' for n in ['ActivePower', 'Freq', 'V_avg']]
        + [f'roll_std_{n}' for n in ['ActivePower', 'Freq', 'V_avg']]
    )
    print(f'Added {len(TEMPORAL_FEATURE_COLS)} temporal derivative/rolling features:')
    print(TEMPORAL_FEATURE_COLS)
else:
    TEMPORAL_FEATURE_COLS = []
    print('USE_TEMPORAL_FEATURES = False -- skipping derivative/rolling features (ablation baseline).')

try:
    full.to_parquet(f'joined_dataset_{PIPELINE_VERSION}.parquet', index=False)
    print(f'\nSaved joined dataset. Shape: {full.shape}')
except ImportError:
    print('\nWARNING: could not save joined_dataset.parquet -- missing pyarrow/fastparquet.')
    print('This file is a convenience intermediate only; nothing downstream in this')
    print('notebook reads it back, so this is safe to ignore and continue. To fix later:')
    print('  conda install -c conda-forge pyarrow')
    print('  (or: pip install pyarrow)')
    print(f'\nContinuing without saving. Shape: {full.shape}')

## 6. Canonical temporal + embargo split

**This is the single split definition used everywhere in this notebook** -- Phase 1's
per-row tabular models, Phase 2's sequence windows, Phase 2's GAT graphs, and the GAT
correlation prior. Applying it once and tagging every row with a `split` column removes
any risk of different sections disagreeing about which timestamps belong to train vs. test.

**Protocol:**
1. Group rows by `(label, PLCID)`.
2. Within each group, sort by `@timestamp`. The first `1 - test_frac` fraction of the
   group's own timeline -> `train`; the last `test_frac` -> `test`.
3. **Embargo:** rows within `EMBARGO_SECONDS` of the boundary (on either side) are dropped.
   This matters because cyber features use a 1-second backward-looking window -- without
   an embargo, a test row just after the boundary and a train row just before it would
   share almost all of the same underlying packets, which is a leakage channel.
4. `StandardScaler` (and every model) only ever sees `train`-tagged rows during fitting.

In [ ]:
FEATURE_COLS = [
    'Freq', 'Theta', 'V_A', 'V_B', 'V_C', 'I_A', 'I_B', 'I_C',
    'ActivePower', 'ReactivePower', 'BreakerStatus',
    'cyber_pkt_count', 'cyber_pktlen_mean', 'cyber_pktlen_std',
    'cyber_n_unique_srcport', 'cyber_n_unique_dstport',
    'cyber_modbus_frac', 'cyber_tls_frac', 'cyber_tcp_frac',
    'cyber_iat_mean', 'cyber_iat_std',
    'voltage_imbalance', 'current_imbalance', 'power_factor', 'freq_deviation',
] + TEMPORAL_FEATURE_COLS + EXT_COLS  # original 43 first (same positions), then the 50 new v6 cyber columns

FEATURE_COLS = FEATURE_COLS[:25]   # 11 raw physical + 10 cyber (1 s) + 4 derived
assert FEATURE_COLS[:11] == ['Freq','Theta','V_A','V_B','V_C','I_A','I_B','I_C','ActivePower','ReactivePower','BreakerStatus'] and all(c.startswith('cyber_') for c in FEATURE_COLS[11:21]) and FEATURE_COLS[21:] == ['voltage_imbalance','current_imbalance','power_factor','freq_deviation']
print(f'Total features used: {len(FEATURE_COLS)} (11 raw physical + 4 derived + 10 cyber @1s)')

LABEL_COL = 'label'
PLC_COL = 'PLCID'
TIME_COL = '@timestamp'

# PIPELINE_VERSION is set in the setup cell (v6)
TEST_FRAC = 0.15
# EMBARGO_SECONDS unchanged from v3 -- the same sweep confirmed 1.0s as a genuine peak
# (not just "not obviously wrong"): performance rises from 0.0s to 1.0s and drops again by
# 2.0s, consistent with 1.0s being exactly wide enough to close the leak (it matches the
# cyber-feature causal window width) with no further benefit, and real cost, from going wider.
EMBARGO_SECONDS = 1.0


def assign_temporal_split(df, label_col=LABEL_COL, plc_col=PLC_COL, time_col=TIME_COL,
                           test_frac=TEST_FRAC, embargo_seconds=EMBARGO_SECONDS):
    df = df.copy()
    split_col = np.empty(len(df), dtype=object)

    for (label, plcid), idx in df.groupby([label_col, plc_col]).groups.items():
        idx = idx.to_numpy()
        order = np.argsort(df.loc[idx, time_col].values)
        idx_sorted = idx[order]
        times_sorted = df.loc[idx_sorted, time_col].values

        n = len(idx_sorted)
        t_min, t_max = times_sorted[0], times_sorted[-1]
        cutoff = t_min + (t_max - t_min) * (1 - test_frac)

        is_test = times_sorted >= cutoff
        is_embargo = (times_sorted >= cutoff - embargo_seconds) & (times_sorted < cutoff + embargo_seconds)

        labels = np.where(is_embargo, 'embargo', np.where(is_test, 'test', 'train'))
        split_col[df.index.get_indexer(idx_sorted)] = labels

    df['split'] = split_col
    return df


full = assign_temporal_split(full)
print(full['split'].value_counts())

n_before = len(full)
full = full[full['split'] != 'embargo'].reset_index(drop=True)
print(f'\nDropped {n_before - len(full)} embargoed rows ({(n_before - len(full))/n_before:.2%})')
print(f'Remaining: {len(full)} rows')

In [ ]:
# Build the feature/label arrays and fit the scaler on train rows only
X = full[FEATURE_COLS].copy()
y = full[LABEL_COL].copy()

nan_cols = X.columns[X.isna().any()].tolist()
if nan_cols:
    print(f'NaN columns found, filling with 0: {nan_cols}')
    X[nan_cols] = X[nan_cols].fillna(0)

train_mask = (full['split'] == 'train').values
test_mask = (full['split'] == 'test').values

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask].reset_index(drop=True), y[test_mask].reset_index(drop=True)
PLCID_train = full.loc[train_mask, PLC_COL].values
PLCID_test = full.loc[test_mask, PLC_COL].values
ts_train = full.loc[train_mask, TIME_COL].values
ts_test = full.loc[test_mask, TIME_COL].values

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Train class dist: {np.bincount(y_train)}')
print(f'Test class dist:  {np.bincount(y_test)}')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6b. Class sizes (reviewer: "give the number of samples in each class")

In [ ]:
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
n_tr = np.bincount(y_train, minlength=6); n_te = np.bincount(y_test, minlength=6)
class_sizes = pd.DataFrame({'class': class_names, 'train_rows': n_tr, 'test_rows': n_te})
class_sizes['total_rows'] = class_sizes.train_rows + class_sizes.test_rows
class_sizes['test_share_%'] = (100 * class_sizes.test_rows / class_sizes.test_rows.sum()).round(2)
print(class_sizes.to_string(index=False))
print(f"\nTrain total {n_tr.sum()}, test total {n_te.sum()}")
class_sizes.to_csv(f'nb1_class_sizes_{PIPELINE_VERSION}.csv', index=False)

## 6c. Dataset-artefact audit (reviewer: "check for dataset artefacts -- distinctive electrical state per attack run")

Each class in this dataset is a **separate recording session**. If sessions differ in their baseline electrical state
(not because of the attack, but because they were recorded at different times / loads), a classifier can learn the
session, not the attack. This section reports (i) when each session was recorded and (ii) the per-class electrical
state, so that any such artefact is visible rather than assumed away. It is evidence for the Limitations section,
not a fix.

In [ ]:
sess = (full.groupby('label_name')[TIME_COL]
        .agg(t_start='min', t_end='max', rows='size')
        .assign(duration_min=lambda d: (d.t_end - d.t_start) / 60.0)
        .loc[class_names])
sess['t_start_utc'] = pd.to_datetime(sess.t_start, unit='s', utc=True).dt.strftime('%Y-%m-%d %H:%M')
sess['t_end_utc']   = pd.to_datetime(sess.t_end,   unit='s', utc=True).dt.strftime('%Y-%m-%d %H:%M')
print('Recording sessions per class (rows kept after embargo):')
print(sess[['rows', 't_start_utc', 't_end_utc', 'duration_min']].round(1).to_string())

overlap = []
for i, a in enumerate(class_names):
    for b in class_names[i+1:]:
        lo = max(sess.loc[a, 't_start'], sess.loc[b, 't_start']); hi = min(sess.loc[a, 't_end'], sess.loc[b, 't_end'])
        overlap.append((a, b, max(0.0, hi - lo) / 60.0))
ov = pd.DataFrame(overlap, columns=['class_a', 'class_b', 'overlap_min'])
print(f"\nPairs of classes whose recording windows overlap in time: {(ov.overlap_min > 0).sum()} of {len(ov)}")
print(ov[ov.overlap_min > 0].round(1).to_string(index=False) if (ov.overlap_min > 0).any()
      else 'None -- every class was recorded in its own, non-overlapping session.')
sess.to_csv(f'nb1_session_audit_{PIPELINE_VERSION}.csv')

phys_state_cols = ['Freq', 'freq_deviation', 'voltage_imbalance', 'current_imbalance', 'power_factor',
                   'ActivePower', 'ReactivePower', 'BreakerStatus']
state = full.groupby('label_name')[phys_state_cols].agg(['mean', 'std']).loc[class_names]
state.columns = [f'{a}_{b}' for a, b in state.columns]
print('\nPer-class physical state (mean, std over all kept rows):')
print(state.round(5).T.to_string())
state.to_csv(f'nb1_physical_state_by_class_{PIPELINE_VERSION}.csv')

# Effect size of each physical feature vs. benign (standardised mean difference) -- artefact screen
benign_rows = full[full.label_name == 'benign']
smd = {}
for c in class_names[1:]:
    rows_c = full[full.label_name == c]
    smd[c] = {col: (rows_c[col].mean() - benign_rows[col].mean()) / (np.sqrt((rows_c[col].var() + benign_rows[col].var()) / 2) + 1e-12)
              for col in phys_state_cols}
smd = pd.DataFrame(smd).T
print('\nStandardised mean difference vs. benign (|d| > 0.8 = large shift in baseline electrical state):')
print(smd.round(2).to_string())
smd.to_csv(f'nb1_physical_smd_vs_benign_{PIPELINE_VERSION}.csv')

## 6c-2. Constant / near-constant feature screen

In [ ]:
rel_std = (full[FEATURE_COLS].std() / (full[FEATURE_COLS].abs().mean() + 1e-12)).sort_values()
const_df = pd.DataFrame({'n_unique': full[FEATURE_COLS].nunique(), 'rel_std': rel_std}).loc[rel_std.index]
print('Features with (near-)zero variance across the WHOLE dataset (relative std < 1e-3):')
print(const_df[const_df.rel_std < 1e-3].to_string())
print('\nFrequency: min/max =', full['Freq'].min(), full['Freq'].max(), '(constant -> freq_deviation, d_Freq, roll_*_Freq carry no information)')
print('BreakerStatus unique values:', full['BreakerStatus'].unique())
const_df.to_csv(f'nb1_constant_feature_screen_{PIPELINE_VERSION}.csv')

## 6d. Timestamp structure and what the embargo actually does (audit finding -- report this)

The physical stream is delivered in **bursts**: each `(class, PLCID)` series has only ~50 distinct `@timestamp`
values, ~5 s apart, with hundreds of rows sharing each one. Consequences that the dissertation should state plainly:
(i) the embargo (+/- 1 s around the cutoff) **drops zero rows** because no timestamp falls inside it -- it is
non-binding, not "validated"; (ii) the split is nevertheless leak-free with respect to the 1 s causal cyber window,
because the last train timestamp and the first test timestamp are >= 1 s apart (checked below);
(iii) the number of *independent* test observations is far smaller than the 97k rows suggest -- rows inside one burst
share an identical timestamp and an identical cyber window.

In [ ]:
rows = []
for (lab, plc), g in full.groupby([LABEL_COL, PLC_COL]):
    tr_t = g.loc[g['split'] == 'train', TIME_COL]; te_t = g.loc[g['split'] == 'test', TIME_COL]
    if len(tr_t) == 0 or len(te_t) == 0:
        rows.append((lab, plc, len(g), g[TIME_COL].nunique(), np.nan, 0, 0)); continue
    rows.append((lab, plc, len(g), g[TIME_COL].nunique(), te_t.min() - tr_t.max(), tr_t.nunique(), te_t.nunique()))
ts_audit = pd.DataFrame(rows, columns=['label', 'PLCID', 'rows', 'distinct_timestamps', 'gap_last_train_to_first_test_s',
                                       'train_timestamps', 'test_timestamps'])
print(f'(class, PLC) groups: {len(ts_audit)}')
print(f'Distinct timestamps per group: median {ts_audit.distinct_timestamps.median():.0f}, '
      f'min {ts_audit.distinct_timestamps.min()}, max {ts_audit.distinct_timestamps.max()}')
print(f'Median rows per distinct timestamp: {(ts_audit.rows / ts_audit.distinct_timestamps).median():.0f}')
mg = ts_audit.gap_last_train_to_first_test_s.min()
print(f'Smallest gap between last TRAIN timestamp and first TEST timestamp over all groups: {mg:.3f} s '
      f'(causal cyber window = 1.0 s)  ->  {"leak-free w.r.t. the cyber window" if mg >= 1.0 else "WINDOWS CAN OVERLAP -- investigate"}')
print(f'Embargo rows dropped: {n_before - len(full)}  (EMBARGO_SECONDS={EMBARGO_SECONDS})')
print(f'Independent test observations (distinct test timestamps summed over groups): {int(ts_audit.test_timestamps.sum())} '
      f'versus {len(y_test)} test rows')
ts_audit.to_csv(f'nb1_timestamp_audit_{PIPELINE_VERSION}.csv', index=False)

## 6e. Feature-set selection check: time-blocked cross-validation inside the TRAIN data only

How the v6 feature set was chosen, reproduced here so it is auditable. Unit of analysis = one distinct
(class, PLC, timestamp) burst (rows within a burst are identical copies), weighted by its row count so the score
equals the row-level weighted F1. Folds are expanding time blocks inside each `(class, PLC)` train series
(fit on 0-55% -> validate 55-70%; fit 0-70% -> 70-85%; fit 0-85% -> 85-100%). **The test split is not touched here.**
The temporal `d_*`/`roll_*` columns are excluded: they are computed over duplicated rows and mostly encode position
inside a burst, not physics.

In [ ]:
_bcols = [c for c in FEATURE_COLS if not c.startswith(('d_', 'roll_'))]
_btr = full[full['split'] == 'train'].copy()
_g = _btr.groupby(['label', 'PLCID', '@timestamp'])
burst = _g[_bcols].first(); burst['rows'] = _g.size(); burst = burst.reset_index()
_gt = burst.groupby(['label', 'PLCID'])['@timestamp']
burst['frac'] = (burst['@timestamp'] - _gt.transform('min')) / (_gt.transform('max') - _gt.transform('min') + 1e-9)
print(f'Train bursts: {len(burst)} (from {len(_btr)} rows)')

_phys = [c for c in _bcols if not c.startswith('cyber_')]
SETS_CV = {
    '25 features (11 raw + 4 derived physical + 10 cyber @1s)': _phys + LEGACY_CYBER,
    'cyber only (10 @1s)':                           LEGACY_CYBER,
    'physical only (11 raw + 4 derived)':            _phys,
}
FOLDS = [(0.0, 0.55, 0.55, 0.70), (0.0, 0.70, 0.70, 0.85), (0.0, 0.85, 0.85, 1.01)]
cv_rows = []
for name, cols in SETS_CV.items():
    scores = []
    for a, b_, c_, d_ in FOLDS:
        fit = burst[(burst.frac >= a) & (burst.frac < b_)]; val = burst[(burst.frac >= c_) & (burst.frac < d_)]
        for sd in SEEDS[:2]:
            m = make_xgb(sd) if 'make_xgb' in globals() else xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8,
                    colsample_bytree=0.8, objective='multi:softprob', num_class=6, random_state=sd, n_jobs=-1, tree_method='hist')
            m.fit(fit[cols], fit['label'], sample_weight=fit['rows'])
            scores.append(f1_score(val['label'], m.predict(val[cols]), average='weighted', sample_weight=val['rows']))
    cv_rows.append({'feature set': name, 'n_features': len(cols), 'time-blocked CV weighted F1': round(float(np.mean(scores)), 4),
                    'std over folds x seeds': round(float(np.std(scores, ddof=1)), 4)})
cv_df = pd.DataFrame(cv_rows); print(cv_df.to_string(index=False))
cv_df.to_csv(f'nb1_featureset_cv_{PIPELINE_VERSION}.csv', index=False)

## 7. Evaluation helpers (shared by all models)

Per run we record: weighted F1 (the headline metric used in every earlier table), macro F1, per-class F1 / recall,
the benign false-positive rate (benign rows flagged as any attack), the attack miss rate (attack rows flagged benign),
and a false-alert rate per PLC-hour on the benign test stream. Alert rate counts every flagged row, so it is an
upper bound: an operator-side alert aggregator would merge consecutive flags.

In [ ]:
benign_test_mask = (y_test.values == 0)
_df_b = pd.DataFrame({'plc': PLCID_test[benign_test_mask], 't': ts_test[benign_test_mask]})
BENIGN_PLC_HOURS = float((_df_b.groupby('plc')['t'].agg(lambda s: s.max() - s.min())).sum() / 3600.0)
print(f'Benign test stream: {benign_test_mask.sum()} rows, {BENIGN_PLC_HOURS:.3f} PLC-hours')


def run_metrics(y_true, y_pred, y_tr_true=None, y_tr_pred=None, benign_plc_hours=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    out = {
        'test_f1': f1_score(y_true, y_pred, average='weighted'),
        'test_macro_f1': f1_score(y_true, y_pred, average='macro'),
        'per_class_f1': f1_score(y_true, y_pred, average=None, labels=range(6), zero_division=0),
    }
    rec = []
    for c in range(6):
        m = (y_true == c)
        rec.append(float((y_pred[m] == c).mean()) if m.any() else np.nan)
    out['per_class_recall'] = np.array(rec)
    b = (y_true == 0); a = ~b
    out['benign_fpr'] = float((y_pred[b] != 0).mean()) if b.any() else np.nan
    out['attack_miss_rate'] = float((y_pred[a] == 0).mean()) if a.any() else np.nan
    if benign_plc_hours:
        out['false_alerts_per_plc_hour'] = float((y_pred[b] != 0).sum() / benign_plc_hours)
    if y_tr_true is not None:
        out['train_f1'] = f1_score(y_tr_true, y_tr_pred, average='weighted')
        out['gap'] = out['train_f1'] - out['test_f1']
    return out


def chance_f1(y_true, y_ref, n_draws=50, seed=0):
    """Expected weighted F1 of a classifier that guesses classes at random according to the TRAIN class
    prior. Anything at or below this level is not a 'sensitive' model, it is a broken or inverted one."""
    rng = np.random.default_rng(seed)
    prior = np.bincount(y_ref, minlength=6) / len(y_ref)
    vals = [f1_score(y_true, rng.choice(6, size=len(y_true), p=prior), average='weighted') for _ in range(n_draws)]
    return float(np.mean(vals))

CHANCE_ROW = chance_f1(y_test.values, y_train.values)
print(f'Chance-level weighted F1 on the row-level test set: {CHANCE_ROW:.4f}')

ALL_RUNS = {}          # ALL_RUNS[model_name][seed] -> metrics dict
def record(model_name, seed, m):
    ALL_RUNS.setdefault(model_name, {})[seed] = m

## 8. Tabular models -- 5 seeds each

XGBoost uses the CPU `hist` method so seed 42 is directly comparable to the earlier v4 baseline (0.7225). GPU XGBoost is
**not** bit-identical to CPU even with the same `random_state`; do not mix backends when comparing across notebooks.
Logistic Regression (lbfgs) is deterministic, so it is fitted once and the result is reused for every seed.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

def make_xgb(seed, n_estimators=200):
    return xgb.XGBClassifier(n_estimators=n_estimators, max_depth=6, learning_rate=0.1, subsample=0.8,
                             colsample_bytree=0.8, objective='multi:softprob', num_class=6,
                             eval_metric='mlogloss', random_state=seed, n_jobs=-1, tree_method='hist')

xgb_models = {}
for seed in SEEDS:
    t0 = time.time()
    m = make_xgb(seed); m.fit(X_train_scaled, y_train, verbose=False)
    xgb_models[seed] = m
    record('XGBoost', seed, run_metrics(y_test, m.predict(X_test_scaled), y_train, m.predict(X_train_scaled), BENIGN_PLC_HOURS))
    r = ALL_RUNS['XGBoost'][seed]
    print(f'XGBoost  seed={seed:<5} train={r["train_f1"]:.4f} test={r["test_f1"]:.4f} gap={r["gap"]:.4f}  ({time.time()-t0:.0f}s)')

ref = ALL_RUNS['XGBoost'][42]['test_f1']
print(f'\nSeed-42 XGBoost test F1 = {ref:.4f}  (earlier v4 CPU run, before the power-factor fix: 0.7225)')
if (not PF_FIX) and abs(ref - 0.7225) > 0.005:
    print('NOTE: differs from the earlier run by >0.005. Check xgboost version / CPU vs GPU backend. '
          'The multiseed mean +/- std below is the number to report either way.')

In [ ]:
for seed in SEEDS:
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=seed, n_jobs=-1)
    rf.fit(X_train_scaled, y_train)
    record('RandomForest', seed, run_metrics(y_test, rf.predict(X_test_scaled), y_train, rf.predict(X_train_scaled), BENIGN_PLC_HOURS))
    r = ALL_RUNS['RandomForest'][seed]
    print(f'RF       seed={seed:<5} train={r["train_f1"]:.4f} test={r["test_f1"]:.4f} gap={r["gap"]:.4f}  ({time.time()-t0:.0f}s)')

lr = LogisticRegression(max_iter=1000, random_state=SEEDS[0], n_jobs=-1)
lr.fit(X_train_scaled, y_train)
lr_m = run_metrics(y_test, lr.predict(X_test_scaled), y_train, lr.predict(X_train_scaled), BENIGN_PLC_HOURS)
for seed in SEEDS:
    record('LogisticRegression', seed, lr_m)
print(f'LogReg   (deterministic) train={lr_m["train_f1"]:.4f} test={lr_m["test_f1"]:.4f} gap={lr_m["gap"]:.4f}')

In [ ]:
import torch.nn as nn

class ClassAutoencoder(nn.Module):
    def __init__(self, n_features, bottleneck=8, hidden=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, hidden), nn.GELU(), nn.Linear(hidden, bottleneck), nn.GELU())
        self.decoder = nn.Sequential(nn.Linear(bottleneck, hidden), nn.GELU(), nn.Linear(hidden, n_features))
    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_class_autoencoder(X_class, n_features, n_epochs=30, seed=42, batch_size=256):
    """Same architecture / optimiser / epochs / batch size as the earlier runs; batching is done on-GPU
    (no DataLoader) purely for speed."""
    torch.manual_seed(seed)
    gen = torch.Generator(device='cpu'); gen.manual_seed(seed)
    model = ClassAutoencoder(n_features).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    Xt = torch.from_numpy(X_class).float().to(DEVICE)
    n = len(Xt)
    for _ in range(n_epochs):
        model.train()
        perm = torch.randperm(n, generator=gen).to(DEVICE)
        for i in range(0, n, batch_size):
            xb = Xt[perm[i:i + batch_size]]
            opt.zero_grad(); loss = crit(model(xb), xb); loss.backward(); opt.step()
    return model


def reconstruction_error(model, X, batch_size=8192):
    model.eval(); errs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            b = torch.from_numpy(X[i:i + batch_size]).float().to(DEVICE)
            errs.append(((model(b) - b) ** 2).mean(dim=1).cpu().numpy())
    return np.concatenate(errs)

n_feat = X_train_scaled.shape[1]
for seed in SEEDS:
    t0 = time.time()
    aes = {c: train_class_autoencoder(X_train_scaled[y_train.values == c], n_feat, seed=seed + c) for c in range(6)}
    te = np.stack([reconstruction_error(aes[c], X_test_scaled) for c in range(6)], axis=1)
    tr = np.stack([reconstruction_error(aes[c], X_train_scaled) for c in range(6)], axis=1)
    record('PerClassAutoencoder', seed, run_metrics(y_test, te.argmin(1), y_train, tr.argmin(1), BENIGN_PLC_HOURS))
    r = ALL_RUNS['PerClassAutoencoder'][seed]
    print(f'AE       seed={seed:<5} train={r["train_f1"]:.4f} test={r["test_f1"]:.4f} gap={r["gap"]:.4f}  ({time.time()-t0:.0f}s)')

## 9. Feature-group ablation -- which features does each class actually depend on?

This is the direct test of the reviewer's concern that "physics immunity" could simply be the threat model (the attacker
may only touch cyber features) rather than a property of the physics. We train XGBoost on (a) **physical-only**
features (everything except the 10 `cyber_*` features: raw telemetry + physics-derived + temporal), (b) **cyber-only**
features, and (c) **all** features, 5 seeds each. If ransomware / reverse-shell are detected almost entirely from the
*physical* group, then a cyber-feature-only attacker cannot move them **by construction of the threat model** and this
must be stated as such in the dissertation; if they are detected mainly from cyber features, the immunity result
needs a different explanation. SHAP in NB8 gives the per-feature view; this section gives the group-level view.

In [ ]:
CYBER_COLS = [c for c in FEATURE_COLS if c.startswith('cyber_')]
PHYS_GROUP = [c for c in FEATURE_COLS if c not in CYBER_COLS]
GROUPS = {'physical_only': PHYS_GROUP, 'cyber_only': CYBER_COLS, 'all_features': list(FEATURE_COLS)}
print({k: len(v) for k, v in GROUPS.items()})

abl = {g: {} for g in GROUPS}
for g, cols in GROUPS.items():
    idx = [FEATURE_COLS.index(c) for c in cols]
    for seed in SEEDS:
        m = make_xgb(seed, n_estimators=200); m.fit(X_train_scaled[:, idx], y_train, verbose=False)
        abl[g][seed] = run_metrics(y_test, m.predict(X_test_scaled[:, idx]), benign_plc_hours=BENIGN_PLC_HOURS)
    print(f'{g:<14} test F1 = {np.mean([abl[g][s]["test_f1"] for s in SEEDS]):.4f} '
          f'+/- {np.std([abl[g][s]["test_f1"] for s in SEEDS], ddof=1):.4f}')

rows = []
for g in GROUPS:
    pcf = np.stack([abl[g][s]['per_class_f1'] for s in SEEDS]); pcr = np.stack([abl[g][s]['per_class_recall'] for s in SEEDS])
    for c, cn in enumerate(class_names):
        rows.append({'feature_group': g, 'class': cn,
                     'F1_mean': pcf[:, c].mean(), 'F1_std': pcf[:, c].std(ddof=1),
                     'recall_mean': pcr[:, c].mean(), 'recall_std': pcr[:, c].std(ddof=1)})
abl_df = pd.DataFrame(rows)
print('\nPer-class F1 (mean over 5 seeds) by feature group:')
print(abl_df.pivot(index='class', columns='feature_group', values='F1_mean').loc[class_names].round(4).to_string())
print('\nPer-class recall (mean over 5 seeds) by feature group:')
print(abl_df.pivot(index='class', columns='feature_group', values='recall_mean').loc[class_names].round(4).to_string())
abl_df.to_csv(f'nb1_feature_group_ablation_{PIPELINE_VERSION}.csv', index=False)

# Gain-based importance share of the cyber group for the seed-42 full model (coarse; SHAP in NB8)
imp = pd.Series(xgb_models[42].feature_importances_, index=FEATURE_COLS)
print(f'\nSeed-42 full model: share of total gain-importance carried by the {len(CYBER_COLS)} cyber features = {imp[CYBER_COLS].sum() / imp.sum():.3f}')

## 10. Phase 2: sequence / graph encoders (SCL) + XGBoost head -- 5 seeds each

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
df = full
print('Shape:', df.shape)
print(df['split'].value_counts())

## Phase 2, Section 2: Build windows

Windows are built **per (label, PLCID) group**, sorted by timestamp, using the `split`
column already assigned in Section 6. Because embargo rows were already dropped from
`full`, the remaining `train` rows within a group are contiguous in time (and likewise
for `test`), so windowing over them cannot straddle the boundary.

In [ ]:
WINDOW = 50
STRIDE = 25


def build_windows(df, window=WINDOW, stride=STRIDE):
    train_X, train_y, test_X, test_y = [], [], [], []

    for (label, plcid), grp in df.groupby([LABEL_COL, PLC_COL]):
        grp = grp.sort_values(TIME_COL)

        for split_name, X_out, y_out in [('train', train_X, train_y),
                                          ('test', test_X, test_y)]:
            block = grp[grp['split'] == split_name]
            feats = block[FEATURE_COLS].values.astype(np.float32)
            n_block = len(feats)
            if n_block < window:
                continue
            for start in range(0, n_block - window + 1, stride):
                X_out.append(feats[start:start + window])
                y_out.append(label)

    return (np.stack(train_X), np.array(train_y),
            np.stack(test_X), np.array(test_y))


X_train_w, y_train_w, X_test_w, y_test_w = build_windows(df)
print('X_train_w:', X_train_w.shape, 'X_test_w:', X_test_w.shape)
print('y_train_w class dist:', np.bincount(y_train_w))
print('y_test_w class dist:', np.bincount(y_test_w))

## Phase 2, Section 3: Supervised Contrastive Loss

In [ ]:
class SupConLoss(nn.Module):
    """Supervised Contrastive Loss (Khosla et al. 2020)."""

    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        sim = torch.matmul(embeddings, embeddings.T) / self.temperature

        labels = labels.view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(embeddings.device)

        eye = torch.eye(mask.shape[0], device=embeddings.device)
        mask = mask - eye
        sim = sim - eye * 1e9

        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)

        pos_count = mask.sum(dim=1)
        valid = pos_count > 0
        loss_per_anchor = -(mask * log_prob).sum(dim=1)[valid] / pos_count[valid]

        return loss_per_anchor.mean()

## Phase 2, Section 4: Encoders -- TCN, LSTM, Transformer

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=self.pad)

    def forward(self, x):
        out = self.conv(x)
        return out[:, :, :-self.pad] if self.pad > 0 else out


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        self.conv1 = CausalConv1d(in_ch, out_ch, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        out = self.act(self.conv1(x))
        out = self.drop(out)
        out = self.act(self.conv2(out))
        out = self.drop(out)
        return self.act(out + res)


class TCNEncoder(nn.Module):
    def __init__(self, n_features, hidden=64, dropout=0.1):
        super().__init__()
        self.block1 = ResBlock(n_features, hidden, kernel_size=3, dilation=1, dropout=dropout)
        self.block2 = ResBlock(hidden, hidden, kernel_size=7, dilation=2, dropout=dropout)
        self.block3 = ResBlock(hidden, hidden, kernel_size=15, dilation=4, dropout=dropout)
        self.proj = nn.Linear(hidden, hidden)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x[:, :, -1]
        return self.proj(x)


class LSTMEncoder(nn.Module):
    def __init__(self, n_features, hidden=64, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.proj = nn.Linear(hidden, hidden)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.proj(h[-1])


class TransformerEncoder(nn.Module):
    def __init__(self, n_features, hidden=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, hidden)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * 2, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.proj = nn.Linear(hidden, hidden)

    def forward(self, x):
        out = self.transformer(self.input_proj(x))
        return self.proj(out.mean(dim=1))

## Phase 2, Section 4b: Graph construction for GAT

**Correlation-derived adjacency is now computed from `split == 'train'` rows only.**
Previously this was computed from the entire dataset (including test), which let
test-period physical telemetry shape the model's attention bias -- a data-leakage
channel independent of any label leakage.

In [ ]:
PHYS_COLS_FOR_CORR = ['Freq', 'V_A', 'V_B', 'V_C', 'I_A', 'I_B', 'I_C', 'ActivePower', 'ReactivePower']
N_PLCS = 9
plcids = list(range(1, N_PLCS + 1))

df_train_only = df[df['split'] == 'train'].copy()
df_train_only['time_bin_corr'] = (df_train_only[TIME_COL] * 10).round() / 10

corr_sum = np.zeros((N_PLCS, N_PLCS))
corr_count = np.zeros((N_PLCS, N_PLCS))

for col in PHYS_COLS_FOR_CORR:
    pivot = df_train_only.pivot_table(index='time_bin_corr', columns=PLC_COL, values=col, aggfunc='mean')
    pivot = pivot.reindex(columns=plcids)
    c = pivot.corr().values
    valid = ~np.isnan(c)
    corr_sum[valid] += np.abs(c[valid])
    corr_count[valid] += 1

corr_matrix = np.divide(corr_sum, corr_count, out=np.zeros_like(corr_sum), where=corr_count > 0)
np.fill_diagonal(corr_matrix, 0)

print('Correlation-derived adjacency (train rows only, |corr| averaged across 9 physical features):')
print(pd.DataFrame(corr_matrix, index=plcids, columns=plcids).round(3))

In [ ]:
TIME_BIN_GRAPH = 5.0


def build_graphs(df, time_bin=TIME_BIN_GRAPH):
    n_feat = len(FEATURE_COLS)
    train_X, train_y, test_X, test_y = [], [], [], []

    for label, grp in df.groupby(LABEL_COL):
        for split_name, X_out, y_out in [('train', train_X, train_y),
                                          ('test', test_X, test_y)]:
            block = grp[grp['split'] == split_name].copy()
            if len(block) == 0:
                continue
            t_min = block[TIME_COL].min()
            block['time_bin'] = ((block[TIME_COL] - t_min) // time_bin).astype(int)

            for b in sorted(block['time_bin'].unique()):
                bin_data = block[block['time_bin'] == b]
                graph_feat = np.zeros((N_PLCS, n_feat), dtype=np.float32)
                for plcid in plcids:
                    node_data = bin_data[bin_data[PLC_COL] == plcid]
                    if len(node_data) > 0:
                        graph_feat[plcid - 1] = node_data[FEATURE_COLS].mean().values
                X_out.append(graph_feat)
                y_out.append(label)

    return (np.stack(train_X), np.array(train_y),
            np.stack(test_X), np.array(test_y))


X_train_g, y_train_g, X_test_g, y_test_g = build_graphs(df)
print('Graph dataset -- X_train:', X_train_g.shape, 'X_test:', X_test_g.shape)
print('y_train class dist:', np.bincount(y_train_g))
print('y_test class dist:', np.bincount(y_test_g))

zero_frac = (X_train_g.sum(axis=2) == 0).mean()
print(f'Fraction of zero (empty) nodes: {zero_frac:.3f}')

## Phase 2, Section 4c: GAT encoder

In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.attn = nn.Linear(2 * out_dim, 1)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x, adj_bias):
        h = self.W(x)
        n = h.shape[1]
        h_i = h.unsqueeze(2).expand(-1, -1, n, -1)
        h_j = h.unsqueeze(1).expand(-1, n, -1, -1)
        cat = torch.cat([h_i, h_j], dim=-1)
        e = self.attn(cat).squeeze(-1)
        e = e + adj_bias.unsqueeze(0)
        alpha = F.softmax(e, dim=-1)
        alpha = self.drop(alpha)
        out = torch.einsum('bij,bjd->bid', alpha, h)
        return self.act(out)


class GATEncoder(nn.Module):
    def __init__(self, n_features, corr_matrix, hidden=64, dropout=0.1):
        super().__init__()
        self.gat1 = GATLayer(n_features, hidden, dropout=dropout)
        self.gat2 = GATLayer(hidden, hidden, dropout=dropout)
        self.proj = nn.Linear(hidden, hidden)

        corr = corr_matrix.copy().astype(np.float32)
        np.fill_diagonal(corr, corr.max() if corr.max() > 0 else 1.0)
        bias = corr * 3.0
        self.register_buffer('adj_bias', torch.from_numpy(bias).float())

    def forward(self, x):
        h = self.gat1(x, self.adj_bias)
        h = self.gat2(h, self.adj_bias)
        graph_emb = h.mean(dim=1)
        return self.proj(graph_emb)

In [ ]:
class GATLayerMasked(nn.Module):
    """GAT layer that ignores absent PLC nodes (all-zero vectors after masked scaling) as attention KEYS."""
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.attn = nn.Linear(2 * out_dim, 1)
        self.act = nn.GELU(); self.drop = nn.Dropout(dropout)

    def forward(self, x, adj_bias, present):               # present: (B, N) bool
        h = self.W(x); n = h.shape[1]
        h_i = h.unsqueeze(2).expand(-1, -1, n, -1); h_j = h.unsqueeze(1).expand(-1, n, -1, -1)
        e = self.attn(torch.cat([h_i, h_j], dim=-1)).squeeze(-1) + adj_bias.unsqueeze(0)
        e = e.masked_fill(~present.unsqueeze(1), -1e9)      # never attend to an absent node
        alpha = self.drop(F.softmax(e, dim=-1))
        return self.act(torch.einsum('bij,bjd->bid', alpha, h))


class GATEncoderMasked(nn.Module):
    def __init__(self, n_features, corr_matrix, hidden=64, dropout=0.1):
        super().__init__()
        self.gat1 = GATLayerMasked(n_features, hidden, dropout); self.gat2 = GATLayerMasked(hidden, hidden, dropout)
        self.proj = nn.Linear(hidden, hidden)
        corr = corr_matrix.copy().astype(np.float32); np.fill_diagonal(corr, corr.max() if corr.max() > 0 else 1.0)
        self.register_buffer('adj_bias', torch.from_numpy(corr * 3.0).float())

    def forward(self, x):
        present = x.abs().sum(dim=-1) > 0
        h = self.gat1(x, self.adj_bias, present); h = self.gat2(h, self.adj_bias, present)
        h = h * present.unsqueeze(-1)
        return self.proj(h.sum(dim=1) / present.sum(dim=1, keepdim=True).clamp(min=1))

## Dataset, training loop and evaluation helpers

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float(); self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


def train_encoder(encoder, train_loader, n_epochs=30, lr=1e-3, verbose=False):
    encoder.to(DEVICE)
    optimizer = torch.optim.Adam(encoder.parameters(), lr=lr)
    criterion = SupConLoss(temperature=0.1)
    for epoch in range(n_epochs):
        encoder.train(); total, nb = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(); loss = criterion(encoder(xb), yb); loss.backward(); optimizer.step()
            total += loss.item(); nb += 1
        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f'    epoch {epoch+1}/{n_epochs}  SCL loss {total/nb:.4f}')
    return encoder


def get_embeddings(encoder, X, batch_size=256):
    encoder.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            out.append(encoder(torch.from_numpy(X[i:i + batch_size]).float().to(DEVICE)).cpu().numpy())
    return np.concatenate(out, axis=0)


def scale_sets(X_train, X_test, mask_empty_nodes=False):
    """Per-feature StandardScaler fitted on train only. With mask_empty_nodes=True (graphs), the scaler is fitted on
    PRESENT nodes only and absent nodes are kept exactly zero after scaling."""
    n_feat = X_train.shape[2]
    tr = X_train.reshape(-1, n_feat); te = X_test.reshape(-1, n_feat)
    sc = StandardScaler()
    if mask_empty_nodes:
        present_tr = np.abs(tr).sum(1) > 0; present_te = np.abs(te).sum(1) > 0
        sc.fit(tr[present_tr])
        tr_s = sc.transform(tr); te_s = sc.transform(te)
        tr_s[~present_tr] = 0.0; te_s[~present_te] = 0.0
    else:
        sc.fit(tr); tr_s = sc.transform(tr); te_s = sc.transform(te)
    return (tr_s.reshape(X_train.shape).astype(np.float32), te_s.reshape(X_test.shape).astype(np.float32))


def evaluate_encoder(name, encoder, X_train, y_train_, X_test, y_test_, seed, n_epochs=30, batch_size=128,
                     mask_empty_nodes=False, verbose=False):
    set_all_seeds(seed)
    X_train_s, X_test_s = scale_sets(X_train, X_test, mask_empty_nodes)
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(WindowDataset(X_train_s, y_train_), batch_size=batch_size, shuffle=True, generator=g)
    encoder = train_encoder(encoder, loader, n_epochs=n_epochs, verbose=verbose)
    emb_tr = get_embeddings(encoder, X_train_s); emb_te = get_embeddings(encoder, X_test_s)
    clf = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, objective='multi:softprob',
                            num_class=6, eval_metric='mlogloss', random_state=seed, n_jobs=-1)
    clf.fit(emb_tr, y_train_, verbose=False)
    m = run_metrics(y_test_, clf.predict(emb_te), y_train_, clf.predict(emb_tr))
    m['test_pred'] = clf.predict(emb_te)
    return m

In [ ]:
N_EPOCHS = 30
n_features = X_train_w.shape[2]
CHANCE_WIN = chance_f1(y_test_w, y_train_w); CHANCE_GRAPH = chance_f1(y_test_g, y_train_g)
print(f'Chance-level weighted F1:  windows {CHANCE_WIN:.4f} | graphs {CHANCE_GRAPH:.4f}')

ENCODERS = [
    ('TCN',         lambda s: TCNEncoder(n_features),                     'win',   False),
    ('LSTM',        lambda s: LSTMEncoder(n_features),                    'win',   False),
    ('Transformer', lambda s: TransformerEncoder(n_features),             'win',   False),
    ('GAT',         lambda s: GATEncoder(n_features, corr_matrix),        'graph', False),   # as in earlier runs
    ('GAT_masked',  lambda s: GATEncoderMasked(n_features, corr_matrix),  'graph', True),    # empty-node fix
]
for name, make, kind, masked in ENCODERS:
    for seed in SEEDS:
        t0 = time.time()
        set_all_seeds(seed)
        enc = make(seed)
        if kind == 'win':
            m = evaluate_encoder(name, enc, X_train_w, y_train_w, X_test_w, y_test_w, seed, N_EPOCHS, 128, False)
        else:
            m = evaluate_encoder(name, enc, X_train_g, y_train_g, X_test_g, y_test_g, seed, N_EPOCHS, 32, masked)
        record(name, seed, m)
        print(f'{name:<12} seed={seed:<5} train={m["train_f1"]:.4f} test={m["test_f1"]:.4f} gap={m["gap"]:.4f} ({time.time()-t0:.0f}s)')

## 11. Summary with variation across seeds

Reported as mean +/- sample std (ddof=1) over 5 seeds, with a 95% t-interval (t(4)=2.776). The Wilcoxon signed-rank
test is **not** used for significance here: with n=5 its smallest attainable two-sided p-value is 2/2^5 = 0.0625, so it
can never reject at 0.05 and a non-significant result would say nothing. Paired differences vs. XGBoost use a paired
t-test over seeds instead.

In [ ]:
T4 = stats.t.ppf(0.975, df=len(SEEDS) - 1)
def arr(model, key): return np.array([ALL_RUNS[model][s][key] for s in SEEDS])

DETERMINISTIC = {'LogisticRegression'}
rows = []
for mname in ALL_RUNS:
    te, tr, gp, ma = arr(mname, 'test_f1'), arr(mname, 'train_f1'), arr(mname, 'gap'), arr(mname, 'test_macro_f1')
    fpr = arr(mname, 'benign_fpr')
    rows.append({'Model': mname,
                 'Test F1 (weighted)': f'{te.mean():.4f} +/- {te.std(ddof=1):.4f}',
                 '95% CI': f'[{te.mean() - T4 * te.std(ddof=1) / np.sqrt(len(te)):.4f}, {te.mean() + T4 * te.std(ddof=1) / np.sqrt(len(te)):.4f}]',
                 'Macro F1': f'{ma.mean():.4f} +/- {ma.std(ddof=1):.4f}',
                 'Train F1': f'{tr.mean():.4f}', 'Gap': f'{gp.mean():.4f} +/- {gp.std(ddof=1):.4f}',
                 'Benign FPR': f'{fpr.mean():.4f}',
                 'seed-invariant': 'yes' if mname in DETERMINISTIC else '',
                 '_mean': te.mean()})
summary = pd.DataFrame(rows).sort_values('_mean', ascending=False).drop(columns='_mean').reset_index(drop=True)
summary.index += 1
print(summary.to_string())
summary.to_csv(f'nb1_main_table_{RUN_TAG}.csv')

print('\nPaired comparison vs XGBoost over the same 5 seeds (positive = other model is better):')
pair_rows = []
for mname in ALL_RUNS:
    if mname == 'XGBoost': continue
    d = arr(mname, 'test_f1') - arr('XGBoost', 'test_f1')
    p = stats.ttest_rel(arr(mname, 'test_f1'), arr('XGBoost', 'test_f1')).pvalue if d.std(ddof=1) > 0 else np.nan
    pair_rows.append({'Model': mname, 'mean diff': d.mean(), 'std diff': d.std(ddof=1), 'paired t p': p})
pair = pd.DataFrame(pair_rows); print(pair.round(4).to_string(index=False))
pair.to_csv(f'nb1_paired_vs_xgboost_{RUN_TAG}.csv', index=False)

## 12. Is XGBoost the best model? (stated explicitly, as the reviewer requested)

In [ ]:
xgb_mean = arr('XGBoost', 'test_f1').mean()
better = [(m, arr(m, 'test_f1').mean()) for m in ALL_RUNS if m != 'XGBoost' and arr(m, 'test_f1').mean() > xgb_mean]
print(f'XGBoost mean test F1 (5 seeds, untuned baseline settings): {xgb_mean:.4f}')
if better:
    print('Models with a HIGHER mean test F1 than XGBoost in this run:')
    for m, v in sorted(better, key=lambda t: -t[1]):
        print(f'   {m:<22} {v:.4f}   (+{v - xgb_mean:.4f})')
    print('\n=> The dissertation/paper must state that XGBoost is NOT the top-scoring model here, and justify the choice')
    print('   on other grounds (explainability via TreeSHAP, inference latency, attack-surface analysis), not on accuracy.')
else:
    print('No other model has a higher mean test F1 than XGBoost in this run.')

print('\nPer-class F1, mean over 5 seeds:')
pc = pd.DataFrame({m: np.stack([ALL_RUNS[m][s]['per_class_f1'] for s in SEEDS]).mean(0) for m in ALL_RUNS}, index=class_names)
print(pc.round(4).T.to_string())
pc.T.to_csv(f'nb1_per_class_f1_{RUN_TAG}.csv')

print('\nFalse alarms on benign test traffic (mean over seeds):')
fa = pd.DataFrame({m: {'benign_FPR': arr(m, 'benign_fpr').mean(), 'attack_miss_rate': arr(m, 'attack_miss_rate').mean(),
                       'false_alerts_per_PLC_hour': (arr(m, 'false_alerts_per_plc_hour').mean()
                                                      if 'false_alerts_per_plc_hour' in ALL_RUNS[m][SEEDS[0]] else np.nan)}
                   for m in ALL_RUNS}).T
print(fa.round(4).to_string()); fa.to_csv(f'nb1_false_alarm_{RUN_TAG}.csv')
print('(Encoder rows use window / graph level samples, so alerts-per-PLC-hour is only defined for row-level models.)')

## 13. GAT diagnosis (reviewer: "F1 0.048 is worse than guessing -- that points to a bug, not sensitivity")

Four checks: (1) chance-level reference for the graph test set; (2) per-class empty-node fraction (absent PLCs are
all-zero vectors that the original code then *scaled* into large non-zero values and *attended to*); (3) a control
that feeds the same graphs, flattened, to XGBoost -- if this scores well, the graphs carry signal and the GAT path is
the problem; (4) the masked GAT variant above.

In [ ]:
print(f'Graph test set class counts : {np.bincount(y_test_g, minlength=6)}')
print(f'Graph train set class counts: {np.bincount(y_train_g, minlength=6)}')
print(f'Chance-level weighted F1 (graphs): {CHANCE_GRAPH:.4f}\n')
print(f'Total test graphs: {len(y_test_g)}  -> each graph is ~{100/len(y_test_g):.1f}% of the test set; weighted F1 on so few samples is very high-variance.\n')

empty_by_class = {class_names[c]: float((X_train_g[y_train_g == c].sum(axis=2) == 0).mean()) for c in range(6) if (y_train_g == c).any()}
print('Fraction of EMPTY nodes per class (train graphs):', {k: round(v, 3) for k, v in empty_by_class.items()})

# Control: flattened graph features -> XGBoost (no GAT)
Xg_tr_s, Xg_te_s = scale_sets(X_train_g, X_test_g, mask_empty_nodes=True)
flat = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, objective='multi:softprob', num_class=6, random_state=42, n_jobs=-1)
flat.fit(Xg_tr_s.reshape(len(Xg_tr_s), -1), y_train_g, verbose=False)
flat_f1 = f1_score(y_test_g, flat.predict(Xg_te_s.reshape(len(Xg_te_s), -1)), average='weighted')
print(f'\nControl -- flattened graph features + XGBoost : test F1 = {flat_f1:.4f}')

g_orig = arr('GAT', 'test_f1'); g_mask = arr('GAT_masked', 'test_f1')
print(f'GAT (original)    : {g_orig.mean():.4f} +/- {g_orig.std(ddof=1):.4f}')
print(f'GAT (masked)      : {g_mask.mean():.4f} +/- {g_mask.std(ddof=1):.4f}')
print(f'Chance            : {CHANCE_GRAPH:.4f}')

if g_orig.mean() <= CHANCE_GRAPH * 1.5:
    print('\n=> Original GAT is at/below chance level: treat as an implementation fault, not "sensitivity".')
if flat_f1 > g_orig.mean() + 0.1:
    print('=> The control shows the graphs DO carry signal, so the fault is in the GAT path (empty-node scaling / attention).')
if g_mask.mean() > g_orig.mean() + 0.05:
    print('=> Masked variant recovers performance: report GAT_masked as the corrected result and disclose the original fault.')
else:
    print('=> Masked variant does not recover performance: the remaining gap is the temporal-split shift on 5-second graph '
          'aggregates (few test graphs per class), which must be reported as a limitation rather than "sensitivity".')
gat_diag = {'chance_graph': CHANCE_GRAPH, 'flat_xgb_f1': flat_f1, 'gat_orig': g_orig, 'gat_masked': g_mask,
            'empty_node_frac_by_class': empty_by_class,
            'graph_test_counts': np.bincount(y_test_g, minlength=6), 'graph_train_counts': np.bincount(y_train_g, minlength=6)}

## 14. Detailed evaluation, seed 42 XGBoost

In [ ]:
m42 = xgb_models[42]; pred42 = m42.predict(X_test_scaled)
print(classification_report(y_test, pred42, target_names=class_names, digits=4))
print(pd.DataFrame(confusion_matrix(y_test, pred42), index=class_names, columns=class_names))
print('\nTop 10 feature importances (seed 42):')
print(pd.Series(m42.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).head(10))

## 15. Save everything to `SmartGrid_Dissertation`

Downstream-compatible files keep their original names (`phase1_data_v4.pkl`, `standard_scaler_v4.pkl`,
`xgb_baseline_model_v4.json` = seed-42 model, `baseline_results_v4.pkl`, `plc_corr_matrix_train_only_v4.pkl`).
`baseline_results_v4.pkl` keeps the old single-seed keys (seed 42) and **adds** `multiseed` (mean/std/per-seed values).

In [ ]:
def ms(model, key='test_f1'):
    a = arr(model, key); return {'mean': float(a.mean()), 'std': float(a.std(ddof=1)), 'per_seed': dict(zip(SEEDS, map(float, a)))}

joblib.dump({
    'PIPELINE_VERSION': PIPELINE_VERSION, 'TEST_FRAC': TEST_FRAC, 'EMBARGO_SECONDS': EMBARGO_SECONDS,
    'X_train_scaled': X_train_scaled, 'X_test_scaled': X_test_scaled,
    'y_train': y_train.values, 'y_test': y_test.values,
    'PLCID_train': PLCID_train, 'PLCID_test': PLCID_test,
    'timestamp_train': ts_train, 'timestamp_test': ts_test, 'FEATURE_COLS': FEATURE_COLS,
    'CYBER_COLS': CYBER_COLS, 'CYBER_IDX': [FEATURE_COLS.index(c) for c in CYBER_COLS],
    'LEGACY_CYBER': LEGACY_CYBER, 'EXT_COLS': EXT_COLS, 'EXT_WINDOWS': EXT_WINDOWS,
}, f'phase1_data_{PIPELINE_VERSION}.pkl')
joblib.dump(scaler, f'standard_scaler_{PIPELINE_VERSION}.pkl')
xgb_models[42].save_model(f'xgb_baseline_model_{PIPELINE_VERSION}.json')
joblib.dump(corr_matrix, f'plc_corr_matrix_train_only_{PIPELINE_VERSION}.pkl')

r42 = lambda m, k: float(ALL_RUNS[m][42][k])
joblib.dump({
    'PIPELINE_VERSION': PIPELINE_VERSION, 'TEST_FRAC': TEST_FRAC, 'EMBARGO_SECONDS': EMBARGO_SECONDS,
    # --- legacy single-seed keys (seed 42), unchanged names ---
    'xgb_train_f1': r42('XGBoost', 'train_f1'), 'xgb_test_f1': r42('XGBoost', 'test_f1'),
    'rf_train_f1': r42('RandomForest', 'train_f1'), 'rf_test_f1': r42('RandomForest', 'test_f1'),
    'lr_train_f1': r42('LogisticRegression', 'train_f1'), 'lr_test_f1': r42('LogisticRegression', 'test_f1'),
    'ae_train_f1': r42('PerClassAutoencoder', 'train_f1'), 'ae_test_f1': r42('PerClassAutoencoder', 'test_f1'),
    # --- multiseed ---
    'SEEDS': SEEDS,
    'multiseed': {m: {'test_f1': ms(m), 'train_f1': ms(m, 'train_f1'), 'gap': ms(m, 'gap'), 'macro_f1': ms(m, 'test_macro_f1'),
                      'benign_fpr': ms(m, 'benign_fpr')} for m in ALL_RUNS},
}, f'baseline_results_{PIPELINE_VERSION}.pkl')

clean_runs = {m: {s: {k: v for k, v in r.items() if k != 'test_pred'} for s, r in d.items()} for m, d in ALL_RUNS.items()}
joblib.dump({'SEEDS': SEEDS, 'ALL_RUNS': clean_runs, 'class_names': class_names, 'class_sizes': class_sizes,
             'feature_group_ablation': abl_df, 'session_audit': sess, 'physical_state_by_class': state,
             'physical_smd_vs_benign': smd, 'gat_diagnosis': gat_diag, 'chance': {'row': CHANCE_ROW, 'win': CHANCE_WIN, 'graph': CHANCE_GRAPH},
             'benign_plc_hours': BENIGN_PLC_HOURS, 'summary_table': summary, 'paired_vs_xgb': pair, 'false_alarm': fa},
            f'nb1_multiseed_results_{RUN_TAG}.pkl')

manifest = {'notebook': 'NB1', 'run_tag': RUN_TAG, 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS,
            'utc_time': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()), 'python': platform.python_version(),
            'xgboost': xgb.__version__, 'torch': torch.__version__, 'device': str(DEVICE),
            'gpu': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
            'n_train': int(len(y_train)), 'n_test': int(len(y_test)), 'n_features': len(FEATURE_COLS)}
json.dump(manifest, open(f'nb1_run_manifest_{RUN_TAG}.json', 'w'), indent=2)

print(f'NB1 ({RUN_TAG}) artifacts in {PROJECT_DIR}:')
for f in [f'phase1_data_{PIPELINE_VERSION}.pkl', f'standard_scaler_{PIPELINE_VERSION}.pkl', f'xgb_baseline_model_{PIPELINE_VERSION}.json',
          f'baseline_results_{PIPELINE_VERSION}.pkl', f'plc_corr_matrix_train_only_{PIPELINE_VERSION}.pkl',
          f'nb1_multiseed_results_{RUN_TAG}.pkl', f'nb1_run_manifest_{RUN_TAG}.json', f'nb1_main_table_{RUN_TAG}.csv',
          f'nb1_paired_vs_xgboost_{RUN_TAG}.csv', f'nb1_per_class_f1_{RUN_TAG}.csv', f'nb1_false_alarm_{RUN_TAG}.csv',
          f'nb1_class_sizes_{PIPELINE_VERSION}.csv', f'nb1_session_audit_{PIPELINE_VERSION}.csv',
          f'nb1_timestamp_audit_{PIPELINE_VERSION}.csv', f'nb1_constant_feature_screen_{PIPELINE_VERSION}.csv', f'nb1_physical_state_by_class_{PIPELINE_VERSION}.csv', f'nb1_physical_smd_vs_benign_{PIPELINE_VERSION}.csv',
          f'nb1_feature_group_ablation_{PIPELINE_VERSION}.csv']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f + (f'  ({os.path.getsize(f)/1e6:.1f} MB)' if os.path.exists(f) else ''))